# 🤖 DRL Trading + VAE — Version Améliorée
## Pipeline complet : Données enrichies → VAE-LSTM → Récompense avancée → PPO vs SAC vs TD3

**Améliorations par rapport à la V1 :**
| Module | V1 | V2 (ce notebook) |
|---|---|---|
| Données | 14 features (OHLCV + indicateurs) | + Fear & Greed + Put/Call Ratio + VIX avancé |
| VAE | Encodeur FC simple | **VAE-LSTM** (fenêtre temporelle 30 jours) |
| Récompense | Variation % portefeuille | **Sharpe Ratio + pénalité drawdown** |
| Agent | PPO seul | **PPO vs SAC vs TD3** |
| Évaluation | Métriques basiques | **Dashboard comparatif complet** |

> ⚠️ **GPU obligatoire** : Exécution → Modifier le type d'exécution → GPU T4


---
# 📦 SECTION 1 — Installation et imports

In [ ]:
!pip install yfinance gymnasium stable-baselines3 torch --quiet

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import gymnasium as gym
from gymnasium import spaces
import warnings, json, os, requests
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports OK | Device : {device}")
if device.type == "cuda":
    print(f"   GPU : {torch.cuda.get_device_name(0)} 🚀")
else:
    print("   ⚠️ Active le GPU : Exécution → Modifier le type d'exécution → GPU T4")


## 💾 Connexion Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/DRL_Trading_Project/v2"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"✅ Drive connecté !")
print(f"   Dossier : {SAVE_DIR}")


---
# 📊 SECTION 2 — Collecte des données enrichies

**Nouvelles sources par rapport à la V1 :**
- 🧠 **Fear & Greed Index** — sentiment du marché (0=peur, 100=euphorie)
- 📉 **Put/Call Ratio** — ratio options baissières/haussières
- 📈 **VIX avancé** — niveaux extrêmes et variations
- 📊 **Données SP500** — corrélation marché global


In [ ]:
import yfinance as yf

# ── Paramètres
TICKER = "SPY"
START  = "2019-01-01"
END    = "2024-01-01"

print("⏳ Téléchargement des données...")

# ── Prix OHLCV
df = yf.download(TICKER, start=START, end=END, auto_adjust=True)
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df = df[["Open","High","Low","Close","Volume"]].copy()
df.dropna(inplace=True)

close = df["Close"]
high  = df["High"]
low   = df["Low"]
vol   = df["Volume"]

# ── INDICATEURS TECHNIQUES (identiques V1)
delta    = close.diff()
gain     = delta.clip(lower=0)
loss     = -delta.clip(upper=0)
avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()
df["RSI"] = 100 - (100 / (1 + avg_gain / (avg_loss + 1e-9)))

ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
df["MACD"]     = ema12 - ema26
df["MACD_sig"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["MACD_hist"]= df["MACD"] - df["MACD_sig"]

sma20 = close.rolling(20).mean()
std20 = close.rolling(20).std()
df["BB_width"] = (sma20 + 2*std20 - (sma20 - 2*std20)) / (sma20 + 1e-9)

tr = pd.concat([(high-low), (high-close.shift()).abs(), (low-close.shift()).abs()], axis=1).max(axis=1)
df["ATR"] = tr.rolling(14).mean()

df["EMA10_ratio"]  = close / close.ewm(span=10,  adjust=False).mean() - 1
df["EMA50_ratio"]  = close / close.ewm(span=50,  adjust=False).mean() - 1
df["EMA200_ratio"] = close / close.ewm(span=200, adjust=False).mean() - 1

df["Return_1d"]  = close.pct_change(1)
df["Return_5d"]  = close.pct_change(5)
df["Return_20d"] = close.pct_change(20)
df["Volume_ratio"] = vol / (vol.rolling(20).mean() + 1e-9)

# ── VIX
vix = yf.download("^VIX", start=START, end=END, auto_adjust=True)
vix.columns = [c[0] if isinstance(c, tuple) else c for c in vix.columns]
df["VIX"] = vix["Close"].reindex(df.index).ffill()
df["VIX_zscore"] = (df["VIX"] - df["VIX"].rolling(60).mean()) / (df["VIX"].rolling(60).std() + 1e-9)
df["VIX_change"] = df["VIX"].pct_change()                      # NOUVEAU : variation du VIX
df["VIX_extreme"] = (df["VIX"] > 30).astype(float)             # NOUVEAU : signal peur extrême

# ── Fear & Greed synthétique (calculé depuis les données disponibles)
# Approximation basée sur : RSI, VIX inversé, momentum, BB position
rsi_norm   = df["RSI"] / 100
vix_inv    = 1 - (df["VIX"] - df["VIX"].min()) / (df["VIX"].max() - df["VIX"].min() + 1e-9)
mom        = (close.pct_change(20) + 0.5).clip(0, 1)
bb_pos     = (close - (sma20 - 2*std20)) / (4*std20 + 1e-9)
bb_pos     = bb_pos.clip(0, 1)

df["Fear_Greed"] = (rsi_norm * 0.25 + vix_inv * 0.35 + mom * 0.25 + bb_pos * 0.15)
df["Fear_Greed"] = df["Fear_Greed"].rolling(5).mean()           # lissage

# ── Put/Call Ratio synthétique (approximé via VIX et volume)
df["PutCall_proxy"] = (df["VIX"] / 20) * (1 + df["Volume_ratio"].clip(0, 3) * 0.2)
df["PutCall_proxy"] = df["PutCall_proxy"].rolling(5).mean()

# ── Corrélation avec QQQ (tech) pour contexte macro
qqq = yf.download("QQQ", start=START, end=END, auto_adjust=True)
qqq.columns = [c[0] if isinstance(c, tuple) else c for c in qqq.columns]
df["QQQ_corr"] = close.pct_change().rolling(20).corr(qqq["Close"].reindex(df.index).pct_change())

df.dropna(inplace=True)

print(f"✅ Dataset enrichi : {len(df)} jours × {len(df.columns)} colonnes")
print(f"   Nouvelles features : VIX_change, VIX_extreme, Fear_Greed, PutCall_proxy, QQQ_corr")


## 🔢 Normalisation des features

In [ ]:
EXCLUDE = ["Open","High","Low","Close","Volume","VIX","BB_upper","BB_lower",
           "EMA_10","EMA_50","EMA_200"]

FEATURES = [c for c in df.columns if c not in EXCLUDE]
print(f"✅ {len(FEATURES)} features pour le VAE-LSTM :")
for f in FEATURES:
    print(f"   · {f}")

df_feat  = df[FEATURES].copy()
feat_min = df_feat.min()
feat_max = df_feat.max()
df_norm  = (df_feat - feat_min) / (feat_max - feat_min + 1e-9)

prices = df["Close"].values.astype(np.float32)
X = df_norm.values.astype(np.float32)

# Sauvegarde
df.to_csv(f"{SAVE_DIR}/raw_data_v2.csv")
df_norm.to_csv(f"{SAVE_DIR}/features_v2.csv")
np.save(f"{SAVE_DIR}/features_array_v2.npy", X)
np.save(f"{SAVE_DIR}/prices_v2.npy", prices)
pd.DataFrame({"min": feat_min, "max": feat_max}).to_csv(f"{SAVE_DIR}/norm_params_v2.csv")

print(f"\n✅ Sauvegardé dans Drive | Shape : {X.shape}")


## 📈 Visualisation du sentiment

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), facecolor='#0f0f0f')

for ax in axes:
    ax.set_facecolor('#1a1a1a')
    for s in ['top','right']: ax.spines[s].set_visible(False)
    for s in ['bottom','left']: ax.spines[s].set_color('#333')
    ax.tick_params(colors='white')

axes[0].plot(df.index, df["Close"], color='#00d4ff', lw=1.2)
axes[0].set_title("Prix SPY", color='white', fontsize=11)
axes[0].set_ylabel("Prix ($)", color='white')

axes[1].plot(df.index, df["Fear_Greed"]*100, color='#ffd700', lw=1.2)
axes[1].axhline(70, color='#ff4444', ls='--', alpha=0.6, lw=0.8, label='Euphorie (70)')
axes[1].axhline(30, color='#00ff88', ls='--', alpha=0.6, lw=0.8, label='Peur (30)')
axes[1].set_title("Fear & Greed Index (synthétique)", color='white', fontsize=11)
axes[1].set_ylabel("Score (0-100)", color='white')
axes[1].legend(facecolor='#333', labelcolor='white', fontsize=8)

axes[2].plot(df.index, df["VIX"], color='#ff6b6b', lw=1.2, label='VIX')
axes[2].fill_between(df.index, df["VIX"], 30,
    where=df["VIX"]>30, color='#ff4444', alpha=0.3, label='Peur extrême')
axes[2].set_title("VIX — Indice de volatilité", color='white', fontsize=11)
axes[2].set_ylabel("VIX", color='white')
axes[2].legend(facecolor='#333', labelcolor='white', fontsize=8)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/sentiment_data.png", dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print("✅ Graphique sentiment sauvegardé !")


---
# 🧠 SECTION 3 — VAE-LSTM (Amélioration majeure)

**Différence avec V1 :**
- V1 : le VAE encode **1 jour** → vecteur z
- V2 : le VAE-LSTM encode **30 jours consécutifs** → vecteur z

Le LSTM capture les **dépendances temporelles** : tendances, régimes de marché, patterns récurrents.

```
[Jour t-29, ..., Jour t-1, Jour t]  (séquence de 30 jours)
              ↓
         LSTM Encoder
              ↓
         µ, σ (8 dims)
              ↓
         z = µ + σ·ε
              ↓
         LSTM Decoder
              ↓
      Reconstruction (30 jours)
```


In [ ]:
# ── Paramètres
SEQ_LEN    = 30     # fenêtre temporelle (30 jours)
INPUT_DIM  = X.shape[1]
LATENT_DIM = 16     # plus grand qu'en V1 car plus de données
HIDDEN_DIM = 128

print(f"✅ Paramètres VAE-LSTM :")
print(f"   Fenêtre temporelle : {SEQ_LEN} jours")
print(f"   Dimension entrée   : {INPUT_DIM} features")
print(f"   Dimension latente  : {LATENT_DIM}")
print(f"   Hidden LSTM        : {HIDDEN_DIM}")


In [ ]:
class VAE_LSTM(nn.Module):
    """
    VAE avec encodeur et décodeur LSTM.
    Entrée : séquence de SEQ_LEN jours × INPUT_DIM features
    Sortie : vecteur latent z de LATENT_DIM dimensions
    """
    def __init__(self, input_dim, latent_dim, hidden_dim, seq_len):
        super().__init__()
        self.seq_len    = seq_len
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        # ── ENCODER LSTM
        self.encoder_lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

        # ── DECODER LSTM
        self.fc_decode  = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        self.fc_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        """x : (batch, seq_len, input_dim)"""
        _, (h, _) = self.encoder_lstm(x)
        h_last = h[-1]                      # dernière couche hidden
        return self.fc_mu(h_last), self.fc_log_var(h_last)

    def reparametrize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        """z : (batch, latent_dim) → reconstruction (batch, seq_len, input_dim)"""
        h0 = self.fc_decode(z).unsqueeze(0).repeat(2, 1, 1)
        c0 = torch.zeros_like(h0)
        dec_input = h0[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.decoder_lstm(dec_input, (h0, c0))
        return torch.sigmoid(self.fc_out(out))

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparametrize(mu, log_var)
        x_recon = self.decode(z)
        return x_recon, mu, log_var

    def get_latent(self, x):
        with torch.no_grad():
            mu, log_var = self.encode(x)
            return self.reparametrize(mu, log_var)


vae = VAE_LSTM(INPUT_DIM, LATENT_DIM, HIDDEN_DIM, SEQ_LEN).to(device)
total_params = sum(p.numel() for p in vae.parameters())
print("✅ VAE-LSTM créé !")
print(vae)
print(f"\n   Paramètres : {total_params:,}")


## 🔀 Préparation des séquences temporelles

In [ ]:
def make_sequences(X, seq_len):
    """Crée des séquences glissantes de longueur seq_len"""
    seqs = []
    for i in range(len(X) - seq_len):
        seqs.append(X[i:i+seq_len])
    return np.array(seqs, dtype=np.float32)

X_seq = make_sequences(X, SEQ_LEN)
prices_seq = prices[SEQ_LEN:]    # prix alignés avec les séquences

print(f"✅ Séquences créées : {X_seq.shape}")
print(f"   → {X_seq.shape[0]} séquences de {SEQ_LEN} jours × {INPUT_DIM} features")

SPLIT  = 0.8
n_train = int(len(X_seq) * SPLIT)

X_train_seq = X_seq[:n_train]
X_test_seq  = X_seq[n_train:]
prices_train = prices_seq[:n_train]
prices_test  = prices_seq[n_train:]

X_train_t = torch.FloatTensor(X_train_seq).to(device)
X_test_t  = torch.FloatTensor(X_test_seq).to(device)

BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(X_train_t, X_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, X_test_t),  batch_size=BATCH_SIZE, shuffle=False)

print(f"\n   Train : {len(X_train_seq)} séquences")
print(f"   Test  : {len(X_test_seq)} séquences")


## 🏋️ Entraînement du VAE-LSTM

In [ ]:
EPOCHS_VAE = 80
LR_VAE     = 1e-3
optimizer_vae = optim.Adam(vae.parameters(), lr=LR_VAE)

def vae_loss_fn(x_recon, x, mu, log_var):
    recon = nn.functional.mse_loss(x_recon, x, reduction='sum')
    kl    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon + kl, recon, kl

train_losses, test_losses = [], []
print(f"🚀 Entraînement VAE-LSTM sur {EPOCHS_VAE} epochs...")
print("-" * 55)

for epoch in range(1, EPOCHS_VAE + 1):
    vae.train()
    total_train = 0
    for bx, _ in train_loader:
        optimizer_vae.zero_grad()
        xr, mu, lv = vae(bx)
        loss, _, _ = vae_loss_fn(xr, bx, mu, lv)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        optimizer_vae.step()
        total_train += loss.item()

    vae.eval()
    total_test = 0
    with torch.no_grad():
        for bx, _ in test_loader:
            xr, mu, lv = vae(bx)
            loss, _, _ = vae_loss_fn(xr, bx, mu, lv)
            total_test += loss.item()

    train_losses.append(total_train / len(X_train_seq))
    test_losses.append(total_test  / len(X_test_seq))

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{EPOCHS_VAE} | Train: {train_losses[-1]:.4f} | Test: {test_losses[-1]:.4f}")

print("-" * 55)
print(f"✅ VAE-LSTM entraîné ! Loss finale test : {test_losses[-1]:.4f}")


In [ ]:
# ── Génération des vecteurs latents Z pour tout le dataset
vae.eval()
Z_all = []
with torch.no_grad():
    for i in range(0, len(X_seq), BATCH_SIZE):
        batch = torch.FloatTensor(X_seq[i:i+BATCH_SIZE]).to(device)
        z = vae.get_latent(batch)
        Z_all.append(z.cpu().numpy())

Z_all = np.concatenate(Z_all, axis=0)
Z_train_v2 = Z_all[:n_train]
Z_test_v2  = Z_all[n_train:]

print(f"✅ Vecteurs latents Z générés :")
print(f"   Shape total  : {Z_all.shape}")
print(f"   Z_train      : {Z_train_v2.shape}")
print(f"   Z_test       : {Z_test_v2.shape}")

# Courbes de loss
fig, ax = plt.subplots(figsize=(12, 4), facecolor='#0f0f0f')
ax.set_facecolor('#1a1a1a')
ax.plot(train_losses, color='#00d4ff', lw=2, label='Train')
ax.plot(test_losses,  color='#ff6b6b', lw=2, ls='--', label='Test')
ax.set_title("Courbes de Loss — VAE-LSTM", color='white', fontsize=12)
ax.set_xlabel("Epoch", color='white')
ax.set_ylabel("Loss", color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#333', labelcolor='white')
for s in ['top','right']: ax.spines[s].set_visible(False)
for s in ['bottom','left']: ax.spines[s].set_color('#333')
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/vaelstm_loss.png", dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

# Sauvegarde
torch.save(vae.state_dict(), f"{SAVE_DIR}/vae_lstm_model.pth")
np.save(f"{SAVE_DIR}/Z_train_v2.npy", Z_train_v2)
np.save(f"{SAVE_DIR}/Z_test_v2.npy",  Z_test_v2)
np.save(f"{SAVE_DIR}/prices_train_v2.npy", prices_train)
np.save(f"{SAVE_DIR}/prices_test_v2.npy",  prices_test)
print("✅ VAE-LSTM et vecteurs Z sauvegardés dans Drive !")


---
# 🎮 SECTION 4 — Environnement de trading amélioré

**Améliorations par rapport à la V1 :**
- ✅ **Récompense Sharpe** : maximise le rendement ajusté au risque
- ✅ **Pénalité drawdown** : punit les pertes importantes
- ✅ **Actions continues** : allouer 0% à 100% du capital (pas juste tout ou rien)
- ✅ **Stop-loss automatique** : coupe les positions en cas de perte > 5%


In [ ]:
class TradingEnvV2(gym.Env):
    """
    Environnement de trading amélioré V2.
    - Actions continues : fraction du capital à investir [0, 1]
    - Récompense : Sharpe ratio glissant + pénalité drawdown
    - Stop-loss automatique à -5%
    """
    def __init__(self, Z, prices, initial_capital=10_000,
                 transaction_cost=0.001, max_drawdown_pct=0.15):
        super().__init__()
        self.Z                = Z.astype(np.float32)
        self.prices           = prices.astype(np.float32)
        self.initial_capital  = initial_capital
        self.transaction_cost = transaction_cost
        self.max_dd_pct       = max_drawdown_pct
        self.n_steps          = len(Z)

        # Actions CONTINUES : fraction [0, 1] du capital à investir
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)

        # Observation : z (LATENT_DIM) + infos portefeuille (5)
        obs_dim = Z.shape[1] + 5
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_idx      = 0
        self.capital       = float(self.initial_capital)
        self.shares_held   = 0.0
        self.total_value   = float(self.initial_capital)
        self.peak_value    = float(self.initial_capital)
        self.trade_history = []
        self.value_history = [self.total_value]
        self.daily_returns = []
        return self._obs(), {}

    def _obs(self):
        z          = self.Z[self.step_idx]
        cap_norm   = self.capital / self.initial_capital
        pos        = self.shares_held * self.prices[self.step_idx] / (self.total_value + 1e-9)
        price_norm = self.prices[self.step_idx] / self.prices[0]
        dd         = (self.total_value - self.peak_value) / (self.peak_value + 1e-9)
        sharpe_proxy = np.mean(self.daily_returns[-20:]) / (np.std(self.daily_returns[-20:]) + 1e-9) if len(self.daily_returns) >= 5 else 0.0
        return np.concatenate([z, [cap_norm, pos, price_norm, dd, sharpe_proxy]])

    def step(self, action):
        price      = self.prices[self.step_idx]
        prev_value = self.total_value
        target_pos = float(np.clip(action[0], 0, 1))  # fraction cible [0, 1]

        # Valeur totale actuelle
        current_stock_val = self.shares_held * price
        current_pos_ratio = current_stock_val / (self.total_value + 1e-9)

        # Stop-loss : si drawdown > max_dd_pct → vendre tout
        current_dd = (self.total_value - self.peak_value) / (self.peak_value + 1e-9)
        if current_dd < -self.max_dd_pct:
            target_pos = 0.0

        # Ajustement de la position
        diff = target_pos - current_pos_ratio
        if abs(diff) > 0.05:    # seuil minimum pour éviter micro-trades
            if diff > 0:        # ACHETER
                cash_to_invest = self.capital * diff / (current_pos_ratio + 1e-9) if current_pos_ratio > 0 else self.capital * diff
                cash_to_invest = min(cash_to_invest, self.capital)
                shares_to_buy  = cash_to_invest * (1 - self.transaction_cost) / (price + 1e-9)
                self.shares_held += shares_to_buy
                self.capital     -= cash_to_invest
                self.trade_history.append(("BUY", self.step_idx, price, target_pos))
            else:               # VENDRE
                shares_to_sell  = self.shares_held * abs(diff) / (current_pos_ratio + 1e-9)
                shares_to_sell  = min(shares_to_sell, self.shares_held)
                self.capital   += shares_to_sell * price * (1 - self.transaction_cost)
                self.shares_held-= shares_to_sell
                self.trade_history.append(("SELL", self.step_idx, price, target_pos))

        self.capital = max(self.capital, 0)
        self.total_value = self.capital + self.shares_held * price
        self.peak_value  = max(self.peak_value, self.total_value)
        self.value_history.append(self.total_value)

        daily_ret = (self.total_value - prev_value) / (prev_value + 1e-9)
        self.daily_returns.append(daily_ret)

        # ── RÉCOMPENSE AMÉLIORÉE
        # 1. Rendement journalier
        reward = daily_ret * 100

        # 2. Bonus Sharpe glissant (20 jours)
        if len(self.daily_returns) >= 20:
            window  = np.array(self.daily_returns[-20:])
            sharpe  = window.mean() / (window.std() + 1e-8) * np.sqrt(252)
            reward += sharpe * 0.01

        # 3. Pénalité drawdown
        if current_dd < -0.05:
            reward += current_dd * 10   # pénalité croissante

        self.step_idx += 1
        done = self.step_idx >= self.n_steps - 1

        return self._obs(), reward, done, False, {
            "total_value": self.total_value,
            "drawdown": current_dd,
            "position": target_pos,
        }

    def get_portfolio_return(self):
        return (self.total_value - self.initial_capital) / self.initial_capital * 100

    def get_sharpe(self):
        if len(self.daily_returns) < 2: return 0.0
        r = np.array(self.daily_returns)
        return float(np.sqrt(252) * r.mean() / (r.std() + 1e-9))

    def get_max_drawdown(self):
        vals = np.array(self.value_history)
        peak = np.maximum.accumulate(vals)
        return float(((vals - peak) / (peak + 1e-9)).min() * 100)


print("✅ TradingEnvV2 défini !")
print("   Actions : continues [0, 1] (fraction du capital)")
print("   Récompense : rendement + Sharpe bonus + pénalité drawdown")
print("   Stop-loss : automatique si drawdown > 15%")


---
# 🤖 SECTION 5 — Agent PPO (version améliorée)

PPO adapté aux **actions continues** (allocation de capital en %).


In [ ]:
OBS_DIM = LATENT_DIM + 5

class ActorCriticContinuous(nn.Module):
    """Actor-Critic pour actions continues (allocation [0,1])"""
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
        )
        # Actor : mu et log_std pour distribution gaussienne
        self.actor_mu     = nn.Linear(hidden//2, 1)
        self.actor_log_std= nn.Parameter(torch.zeros(1))
        self.critic       = nn.Linear(hidden//2, 1)

    def forward(self, x):
        h    = self.shared(x)
        mu   = torch.sigmoid(self.actor_mu(h))  # [0, 1]
        std  = self.actor_log_std.exp().expand_as(mu)
        val  = self.critic(h)
        return mu, std, val

    def get_action(self, obs):
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        mu, std, val = self(obs_t)
        dist   = torch.distributions.Normal(mu, std)
        action = dist.sample().clamp(0, 1)
        log_p  = dist.log_prob(action).sum(-1)
        return action.cpu().numpy()[0], log_p, val.squeeze()


def compute_gae(rewards, values, gamma=0.95, lam=0.95):
    """Generalized Advantage Estimation"""
    advantages = []
    gae = 0
    for r, v in zip(reversed(rewards), reversed(values)):
        delta = r - v.item()
        gae   = delta + gamma * lam * gae
        advantages.insert(0, gae)
    advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    returns    = advantages + torch.stack(values).detach()
    return advantages, returns


print("✅ ActorCriticContinuous défini !")
print(f"   OBS_DIM : {OBS_DIM}")


In [ ]:
def train_ppo(Z_train, prices_train, n_episodes=300, label="PPO"):
    """Entraîne un agent PPO et retourne le modèle + historique"""
    model     = ActorCriticContinuous(OBS_DIM).to(device)
    optimizer = optim.Adam(model.parameters(), lr=3e-4)
    env       = TradingEnvV2(Z_train, prices_train)

    CLIP_EPS     = 0.15
    ENTROPY_COEF = 0.01
    VALUE_COEF   = 0.5
    N_PPO_EPOCHS = 4
    BATCH        = 64

    ep_returns  = []
    best_ret    = -np.inf
    best_state  = None

    print(f"🚀 Entraînement {label} sur {n_episodes} épisodes...")
    print("-" * 50)

    for ep in range(1, n_episodes+1):
        obs, _ = env.reset()
        obs_buf, act_buf, rew_buf, lp_buf, val_buf = [], [], [], [], []
        done = False

        while not done:
            action, lp, val = model.get_action(obs)
            next_obs, rew, done, _, _ = env.step(action)
            obs_buf.append(obs)
            act_buf.append(action)
            rew_buf.append(rew)
            lp_buf.append(lp)
            val_buf.append(val)
            obs = next_obs

        advantages, returns = compute_gae(rew_buf, val_buf)
        obs_t  = torch.FloatTensor(np.array(obs_buf)).to(device)
        act_t  = torch.FloatTensor(np.array(act_buf)).to(device)
        old_lp = torch.stack(lp_buf).detach()

        for _ in range(N_PPO_EPOCHS):
            idx = torch.randperm(len(obs_t))
            for s in range(0, len(obs_t), BATCH):
                b = idx[s:s+BATCH]
                mu, std, vals = model(obs_t[b])
                dist  = torch.distributions.Normal(mu, std)
                new_lp= dist.log_prob(act_t[b].unsqueeze(-1)).sum(-1)
                ratio = (new_lp - old_lp[b]).exp()
                adv   = advantages[b]
                surr1 = ratio * adv
                surr2 = ratio.clamp(1-CLIP_EPS, 1+CLIP_EPS) * adv
                a_loss= -torch.min(surr1, surr2).mean()
                c_loss= nn.functional.mse_loss(vals.squeeze(), returns[b])
                ent   = dist.entropy().mean()
                loss  = a_loss + VALUE_COEF*c_loss - ENTROPY_COEF*ent
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer.step()

        ret = env.get_portfolio_return()
        ep_returns.append(ret)
        if ret > best_ret:
            best_ret   = ret
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if ep % 50 == 0 or ep == 1:
            avg = np.mean(ep_returns[-20:])
            print(f"  Ep {ep:4d}/{n_episodes} | Moy: {avg:+.2f}% | Best: {best_ret:+.2f}%")

    model.load_state_dict(best_state)
    print(f"\n✅ {label} terminé | Meilleur rendement train : {best_ret:+.2f}%")
    return model, ep_returns


ppo_model, ppo_history = train_ppo(Z_train_v2, prices_train, n_episodes=300, label="PPO")
torch.save(ppo_model.state_dict(), f"{SAVE_DIR}/ppo_v2.pth")
print("✅ PPO V2 sauvegardé !")


---
# 🔥 SECTION 6 — Agent SAC (Soft Actor-Critic)

**Pourquoi SAC est meilleur que PPO pour le trading ?**
- SAC maximise **à la fois** le rendement ET l'entropie (exploration)
- Il utilise **2 Critics** (Twin Critics) pour éviter la surestimation
- Il apprend **en ligne** (off-policy) → utilise mieux chaque expérience
- Naturellement adapté aux **actions continues**


In [ ]:
class SACCritic(nn.Module):
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        # Twin Q-networks
        self.q1 = nn.Sequential(
            nn.Linear(obs_dim+1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        self.q2 = nn.Sequential(
            nn.Linear(obs_dim+1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, obs, action):
        x = torch.cat([obs, action], dim=-1)
        return self.q1(x), self.q2(x)


class SACActor(nn.Module):
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
        )
        self.mu_head      = nn.Linear(hidden, 1)
        self.log_std_head = nn.Linear(hidden, 1)

    def forward(self, obs):
        h       = self.net(obs)
        mu      = self.mu_head(h)
        log_std = self.log_std_head(h).clamp(-4, 2)
        std     = log_std.exp()
        dist    = torch.distributions.Normal(mu, std)
        x_t     = dist.rsample()
        action  = torch.sigmoid(x_t)           # [0, 1]
        log_prob= dist.log_prob(x_t)
        log_prob-= torch.log(action * (1 - action) + 1e-6)
        return action, log_prob.sum(-1, keepdim=True)


class ReplayBuffer:
    def __init__(self, capacity=50_000, obs_dim=OBS_DIM):
        self.cap  = capacity
        self.ptr  = 0
        self.size = 0
        self.obs  = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.act  = np.zeros((capacity, 1),       dtype=np.float32)
        self.rew  = np.zeros((capacity, 1),       dtype=np.float32)
        self.next = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.done = np.zeros((capacity, 1),       dtype=np.float32)

    def add(self, obs, act, rew, nobs, done):
        self.obs[self.ptr]  = obs
        self.act[self.ptr]  = act
        self.rew[self.ptr]  = rew
        self.next[self.ptr] = nobs
        self.done[self.ptr] = done
        self.ptr  = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, batch=256):
        idx  = np.random.randint(0, self.size, batch)
        return (torch.FloatTensor(self.obs[idx]).to(device),
                torch.FloatTensor(self.act[idx]).to(device),
                torch.FloatTensor(self.rew[idx]).to(device),
                torch.FloatTensor(self.next[idx]).to(device),
                torch.FloatTensor(self.done[idx]).to(device))


print("✅ SAC (Actor + Twin Critics + Replay Buffer) défini !")


In [ ]:
def train_sac(Z_train, prices_train, n_episodes=300, label="SAC"):
    actor   = SACActor(OBS_DIM).to(device)
    critic  = SACCritic(OBS_DIM).to(device)
    critic_target = SACCritic(OBS_DIM).to(device)
    critic_target.load_state_dict(critic.state_dict())

    opt_actor  = optim.Adam(actor.parameters(),  lr=3e-4)
    opt_critic = optim.Adam(critic.parameters(), lr=3e-4)

    # Température d'entropie (alpha) — apprise automatiquement
    log_alpha  = torch.zeros(1, requires_grad=True, device=device)
    opt_alpha  = optim.Adam([log_alpha], lr=3e-4)
    target_ent = -1.0

    buffer = ReplayBuffer(capacity=50_000, obs_dim=OBS_DIM)
    env    = TradingEnvV2(Z_train, prices_train)

    GAMMA  = 0.95
    TAU    = 0.005
    BATCH  = 256

    ep_returns = []
    best_ret   = -np.inf
    best_state = None

    print(f"🚀 Entraînement {label} sur {n_episodes} épisodes...")
    print("-" * 50)

    for ep in range(1, n_episodes+1):
        obs, _ = env.reset()
        done   = False

        while not done:
            obs_t  = torch.FloatTensor(obs).unsqueeze(0).to(device)
            with torch.no_grad():
                action, _ = actor(obs_t)
            act_np = action.cpu().numpy()[0]
            next_obs, rew, done, _, _ = env.step(act_np)
            buffer.add(obs, act_np, rew, next_obs, float(done))
            obs = next_obs

            # Mise à jour SAC (si assez de données)
            if buffer.size >= BATCH:
                o, a, r, no, d = buffer.sample(BATCH)

                with torch.no_grad():
                    na, nlp    = actor(no)
                    q1t, q2t   = critic_target(no, na)
                    qt         = torch.min(q1t, q2t) - log_alpha.exp() * nlp
                    target_q   = r + GAMMA * (1 - d) * qt

                q1, q2    = critic(o, a)
                c_loss    = nn.functional.mse_loss(q1, target_q) + nn.functional.mse_loss(q2, target_q)
                opt_critic.zero_grad(); c_loss.backward(); opt_critic.step()

                na2, lp2  = actor(o)
                q1n, q2n  = critic(o, na2)
                a_loss    = (log_alpha.exp() * lp2 - torch.min(q1n, q2n)).mean()
                opt_actor.zero_grad(); a_loss.backward(); opt_actor.step()

                alpha_loss= -(log_alpha * (lp2 + target_ent).detach()).mean()
                opt_alpha.zero_grad(); alpha_loss.backward(); opt_alpha.step()

                for p, pt in zip(critic.parameters(), critic_target.parameters()):
                    pt.data.copy_(TAU*p.data + (1-TAU)*pt.data)

        ret = env.get_portfolio_return()
        ep_returns.append(ret)
        if ret > best_ret:
            best_ret   = ret
            best_state = {k: v.clone() for k, v in actor.state_dict().items()}

        if ep % 50 == 0 or ep == 1:
            avg = np.mean(ep_returns[-20:])
            print(f"  Ep {ep:4d}/{n_episodes} | Moy: {avg:+.2f}% | Best: {best_ret:+.2f}%")

    actor.load_state_dict(best_state)
    print(f"\n✅ {label} terminé | Meilleur rendement train : {best_ret:+.2f}%")
    return actor, ep_returns


sac_actor, sac_history = train_sac(Z_train_v2, prices_train, n_episodes=300, label="SAC")
torch.save(sac_actor.state_dict(), f"{SAVE_DIR}/sac_v2.pth")
print("✅ SAC V2 sauvegardé !")


---
# ⚡ SECTION 7 — Agent TD3 (Twin Delayed DDPG)

**Pourquoi TD3 ?**
- TD3 est connu pour être **très stable** sur les environnements financiers
- Il utilise un **bruit gaussien déterministe** (moins d'exploration que SAC mais plus stable)
- Le **delayed update** : le Actor est mis à jour moins souvent que les Critics → meilleure stabilité
- Excellent pour les **stratégies conservatrices** (faible drawdown)


In [ ]:
class TD3Actor(nn.Module):
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),  nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, hidden//2),nn.ReLU(),
            nn.Linear(hidden//2, 1),
        )
    def forward(self, x):
        return torch.sigmoid(self.net(x))   # [0, 1]


class TD3Critic(nn.Module):
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.q1 = nn.Sequential(
            nn.Linear(obs_dim+1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        self.q2 = nn.Sequential(
            nn.Linear(obs_dim+1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.q1(x), self.q2(x)
    def q1_only(self, obs, act):
        return self.q1(torch.cat([obs, act], dim=-1))


def train_td3(Z_train, prices_train, n_episodes=300, label="TD3"):
    actor         = TD3Actor(OBS_DIM).to(device)
    actor_target  = TD3Actor(OBS_DIM).to(device)
    critic        = TD3Critic(OBS_DIM).to(device)
    critic_target = TD3Critic(OBS_DIM).to(device)
    actor_target.load_state_dict(actor.state_dict())
    critic_target.load_state_dict(critic.state_dict())

    opt_actor  = optim.Adam(actor.parameters(),  lr=3e-4)
    opt_critic = optim.Adam(critic.parameters(), lr=3e-4)

    buffer = ReplayBuffer(capacity=50_000, obs_dim=OBS_DIM)
    env    = TradingEnvV2(Z_train, prices_train)

    GAMMA        = 0.95
    TAU          = 0.005
    POLICY_DELAY = 2
    NOISE        = 0.1
    NOISE_CLIP   = 0.3
    BATCH        = 256

    ep_returns = []
    best_ret   = -np.inf
    best_state = None
    update_cnt = 0

    print(f"🚀 Entraînement {label} sur {n_episodes} épisodes...")
    print("-" * 50)

    for ep in range(1, n_episodes+1):
        obs, _ = env.reset()
        done   = False

        while not done:
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            with torch.no_grad():
                act = actor(obs_t)
                noise = (torch.randn_like(act) * NOISE).clamp(-NOISE_CLIP, NOISE_CLIP)
                act   = (act + noise).clamp(0, 1)
            act_np = act.cpu().numpy()[0]
            next_obs, rew, done, _, _ = env.step(act_np)
            buffer.add(obs, act_np, rew, next_obs, float(done))
            obs = next_obs

            if buffer.size >= BATCH:
                o, a, r, no, d = buffer.sample(BATCH)
                update_cnt += 1

                with torch.no_grad():
                    noise_t = (torch.randn_like(a) * NOISE).clamp(-NOISE_CLIP, NOISE_CLIP)
                    na      = (actor_target(no) + noise_t).clamp(0, 1)
                    q1t, q2t= critic_target(no, na)
                    target_q= r + GAMMA*(1-d)*torch.min(q1t, q2t)

                q1, q2  = critic(o, a)
                c_loss  = nn.functional.mse_loss(q1, target_q) + nn.functional.mse_loss(q2, target_q)
                opt_critic.zero_grad(); c_loss.backward(); opt_critic.step()

                if update_cnt % POLICY_DELAY == 0:
                    a_loss = -critic.q1_only(o, actor(o)).mean()
                    opt_actor.zero_grad(); a_loss.backward(); opt_actor.step()
                    for p, pt in zip(actor.parameters(), actor_target.parameters()):
                        pt.data.copy_(TAU*p.data + (1-TAU)*pt.data)
                    for p, pt in zip(critic.parameters(), critic_target.parameters()):
                        pt.data.copy_(TAU*p.data + (1-TAU)*pt.data)

        ret = env.get_portfolio_return()
        ep_returns.append(ret)
        if ret > best_ret:
            best_ret   = ret
            best_state = {k: v.clone() for k, v in actor.state_dict().items()}

        if ep % 50 == 0 or ep == 1:
            avg = np.mean(ep_returns[-20:])
            print(f"  Ep {ep:4d}/{n_episodes} | Moy: {avg:+.2f}% | Best: {best_ret:+.2f}%")

    actor.load_state_dict(best_state)
    print(f"\n✅ {label} terminé | Meilleur rendement train : {best_ret:+.2f}%")
    return actor, ep_returns


td3_actor, td3_history = train_td3(Z_train_v2, prices_train, n_episodes=300, label="TD3")
torch.save(td3_actor.state_dict(), f"{SAVE_DIR}/td3_v2.pth")
print("✅ TD3 V2 sauvegardé !")


---
# 📊 SECTION 8 — Évaluation comparative : PPO vs SAC vs TD3

Évaluation sur le **test set uniquement** (données jamais vues).


In [ ]:
def evaluate_agent(model, model_type, Z_test, prices_test, label):
    """Évalue un agent sur le test set et retourne toutes les métriques"""
    env = TradingEnvV2(Z_test, prices_test)
    obs, _ = env.reset()
    done = False

    with torch.no_grad():
        while not done:
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            if model_type == "ppo":
                mu, std, _ = model(obs_t)
                action = mu.cpu().numpy()[0]     # greedy
            else:
                action = model(obs_t).cpu().numpy()[0]
            obs, _, done, _, _ = env.step(action)

    ret    = env.get_portfolio_return()
    sharpe = env.get_sharpe()
    mdd    = env.get_max_drawdown()
    vols   = np.std(env.daily_returns) * np.sqrt(252) * 100 if env.daily_returns else 0

    buys  = [p for t, _, p, _ in env.trade_history if t=="BUY"]
    sells = [p for t, _, p, _ in env.trade_history if t=="SELL"]
    wins  = sum(1 for b, s in zip(buys, sells) if s > b)
    wr    = wins/len(buys)*100 if buys else 0
    gains = sum(max(s-b,0) for b,s in zip(buys,sells))
    losses= sum(max(b-s,0) for b,s in zip(buys,sells))
    pf    = gains/(losses+1e-9)
    calmar= ret / abs(mdd) if mdd != 0 else 0

    print(f"\n{'='*55}")
    print(f"  📊 {label} — TEST SET")
    print(f"{'='*55}")
    print(f"  Rendement        : {ret:+.2f}%")
    print(f"  Sharpe Ratio     : {sharpe:.3f}")
    print(f"  Max Drawdown     : {mdd:.2f}%")
    print(f"  Volatilité       : {vols:.2f}%")
    print(f"  Win Rate         : {wr:.1f}%")
    print(f"  Profit Factor    : {pf:.3f}")
    print(f"  Calmar Ratio     : {calmar:.3f}")
    print(f"  Nb Trades        : {len(buys)}")
    print(f"  Capital final    : {env.total_value:,.2f} $")

    return {
        "label": label, "env": env,
        "return": ret, "sharpe": sharpe, "mdd": mdd,
        "vol": vols, "wr": wr, "pf": pf, "calmar": calmar,
        "n_trades": len(buys), "final_value": env.total_value,
        "value_history": env.value_history,
        "trade_history": env.trade_history,
        "daily_returns": env.daily_returns,
    }


# Buy & Hold
bnh_ret  = (prices_test[-1] - prices_test[0]) / prices_test[0] * 100
bnh_vals = prices_test / prices_test[0] * 10_000
bnh_rets = np.diff(prices_test) / prices_test[:-1]
bnh_sharpe = float(np.sqrt(252) * bnh_rets.mean() / (bnh_rets.std() + 1e-9))
bnh_peak   = np.maximum.accumulate(bnh_vals)
bnh_mdd    = float(((bnh_vals - bnh_peak)/(bnh_peak+1e-9)).min() * 100)

print("⏳ Évaluation des 3 agents sur le test set...")

results_ppo = evaluate_agent(ppo_model, "ppo", Z_test_v2, prices_test, "PPO V2")
results_sac = evaluate_agent(sac_actor, "sac", Z_test_v2, prices_test, "SAC")
results_td3 = evaluate_agent(td3_actor, "td3", Z_test_v2, prices_test, "TD3")

print(f"\n{'='*55}")
print(f"  📊 BUY & HOLD (référence)")
print(f"{'='*55}")
print(f"  Rendement        : {bnh_ret:+.2f}%")
print(f"  Sharpe Ratio     : {bnh_sharpe:.3f}")
print(f"  Max Drawdown     : {bnh_mdd:.2f}%")


## 📊 Tableau comparatif complet

In [ ]:
metrics_table = pd.DataFrame({
    "PPO V2":    [results_ppo["return"], results_ppo["sharpe"], results_ppo["mdd"],
                  results_ppo["vol"], results_ppo["wr"], results_ppo["pf"],
                  results_ppo["calmar"], results_ppo["n_trades"], results_ppo["final_value"]],
    "SAC":       [results_sac["return"], results_sac["sharpe"], results_sac["mdd"],
                  results_sac["vol"], results_sac["wr"], results_sac["pf"],
                  results_sac["calmar"], results_sac["n_trades"], results_sac["final_value"]],
    "TD3":       [results_td3["return"], results_td3["sharpe"], results_td3["mdd"],
                  results_td3["vol"], results_td3["wr"], results_td3["pf"],
                  results_td3["calmar"], results_td3["n_trades"], results_td3["final_value"]],
    "Buy & Hold":[bnh_ret, bnh_sharpe, bnh_mdd, float(np.std(bnh_rets)*np.sqrt(252)*100),
                  "N/A", "N/A", bnh_ret/abs(bnh_mdd) if bnh_mdd!=0 else 0, 1, float(bnh_vals[-1])],
}, index=["Rendement (%)", "Sharpe Ratio", "Max Drawdown (%)", "Volatilité (%)",
          "Win Rate (%)", "Profit Factor", "Calmar Ratio", "Nb Trades", "Capital Final ($)"])

print("\n" + "="*70)
print("  📊 TABLEAU COMPARATIF FINAL — TEST SET")
print("="*70)
print(metrics_table.round(3).to_string())
print("="*70)

metrics_table.to_csv(f"{SAVE_DIR}/comparaison_finale.csv")
print("\n✅ Tableau sauvegardé dans Drive !")


## 🎨 Dashboard final comparatif

In [ ]:
colors = {"PPO V2": "#00d4ff", "SAC": "#ffd700", "TD3": "#ff6b6b", "Buy & Hold": "#aaaaaa"}
results_list = [results_ppo, results_sac, results_td3]
labels_list  = ["PPO V2", "SAC", "TD3"]

fig = plt.figure(figsize=(18, 16), facecolor='#0d0d0d')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Portefeuille (ligne entière)
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor('#1a1a1a')
for r in results_list:
    ax1.plot(r["value_history"], color=colors[r["label"]], lw=2, label=f"{r['label']} ({r['return']:+.2f}%)")
ax1.plot(bnh_vals[:len(results_ppo["value_history"])], color=colors["Buy & Hold"],
         lw=1.5, ls='--', label=f"Buy & Hold ({bnh_ret:+.2f}%)")
ax1.axhline(10_000, color='#444', lw=0.8, ls=':')
ax1.set_title("Valeur du portefeuille — PPO vs SAC vs TD3 vs Buy & Hold (TEST SET)",
              color='white', fontsize=13, fontweight='bold')
ax1.set_ylabel("Valeur ($)", color='white')
ax1.tick_params(colors='white')
ax1.legend(facecolor='#222', labelcolor='white', fontsize=10, ncol=4)
for s in ['top','right']: ax1.spines[s].set_visible(False)
for s in ['bottom','left']: ax1.spines[s].set_color('#333')

# ── 2. Rendements comparés (bar chart)
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor('#1a1a1a')
lbls = ["PPO V2", "SAC", "TD3", "B&H"]
vals = [results_ppo["return"], results_sac["return"], results_td3["return"], bnh_ret]
cols = [colors["PPO V2"], colors["SAC"], colors["TD3"], colors["Buy & Hold"]]
bars = ax2.bar(lbls, vals, color=cols, alpha=0.85, width=0.5)
ax2.axhline(0, color='#555', lw=0.8)
for bar, v in zip(bars, vals):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.5, f"{v:+.1f}%",
             ha='center', va='bottom', color='white', fontsize=9)
ax2.set_title("Rendement (%)", color='white', fontsize=11)
ax2.set_ylabel("%", color='white')
ax2.tick_params(colors='white')
for s in ['top','right']: ax2.spines[s].set_visible(False)
for s in ['bottom','left']: ax2.spines[s].set_color('#333')

# ── 3. Sharpe Ratio
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor('#1a1a1a')
sh_vals = [results_ppo["sharpe"], results_sac["sharpe"], results_td3["sharpe"], bnh_sharpe]
bars3   = ax3.bar(lbls, sh_vals, color=cols, alpha=0.85, width=0.5)
ax3.axhline(1, color='#00ff88', lw=1, ls='--', alpha=0.6)
ax3.axhline(0, color='#555', lw=0.8)
for bar, v in zip(bars3, sh_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, v+0.02, f"{v:.2f}",
             ha='center', va='bottom', color='white', fontsize=9)
ax3.set_title("Sharpe Ratio", color='white', fontsize=11)
ax3.tick_params(colors='white')
for s in ['top','right']: ax3.spines[s].set_visible(False)
for s in ['bottom','left']: ax3.spines[s].set_color('#333')

# ── 4. Max Drawdown
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor('#1a1a1a')
dd_vals = [results_ppo["mdd"], results_sac["mdd"], results_td3["mdd"], bnh_mdd]
bars4   = ax4.bar(lbls, dd_vals, color=cols, alpha=0.85, width=0.5)
for bar, v in zip(bars4, dd_vals):
    ax4.text(bar.get_x()+bar.get_width()/2, v-0.5, f"{v:.1f}%",
             ha='center', va='top', color='white', fontsize=9)
ax4.set_title("Max Drawdown (%)", color='white', fontsize=11)
ax4.tick_params(colors='white')
for s in ['top','right']: ax4.spines[s].set_visible(False)
for s in ['bottom','left']: ax4.spines[s].set_color('#333')

# ── 5-7. Distribution des rendements
for i, (r, lbl) in enumerate(zip(results_list, labels_list)):
    ax = fig.add_subplot(gs[2, i])
    ax.set_facecolor('#1a1a1a')
    rets_pct = np.array(r["daily_returns"]) * 100
    ax.hist(rets_pct, bins=35, color=colors[lbl], alpha=0.8, edgecolor='none')
    ax.axvline(np.mean(rets_pct), color='white', lw=1.5, ls='--',
               label=f"Moy: {np.mean(rets_pct):.3f}%")
    ax.set_title(f"Rendements journaliers — {lbl}", color='white', fontsize=10)
    ax.set_xlabel("Rendement (%)", color='white')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#222', labelcolor='white', fontsize=8)
    for s in ['top','right']: ax.spines[s].set_visible(False)
    for s in ['bottom','left']: ax.spines[s].set_color('#333')

fig.suptitle("🤖 DRL Trading VAE V2 — Dashboard Comparatif Final",
             color='white', fontsize=15, fontweight='bold', y=0.98)

plt.savefig(f"{SAVE_DIR}/dashboard_comparatif.png", dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("✅ Dashboard comparatif sauvegardé dans Drive !")


## 📈 Courbes d'entraînement comparées

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0f0f0f')
ax.set_facecolor('#1a1a1a')

window = 20
for hist, lbl in [(ppo_history, "PPO V2"), (sac_history, "SAC"), (td3_history, "TD3")]:
    smoothed = pd.Series(hist).rolling(window).mean()
    ax.plot(hist,     color=colors[lbl], alpha=0.2, lw=0.8)
    ax.plot(smoothed, color=colors[lbl], lw=2, label=f"{lbl} (moy. {window} ep.)")

ax.axhline(0, color='#444', lw=0.8, ls='--')
ax.set_title("Rendement par épisode (train) — PPO vs SAC vs TD3",
             color='white', fontsize=13, fontweight='bold')
ax.set_xlabel("Épisode", color='white')
ax.set_ylabel("Rendement (%)", color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#222', labelcolor='white', fontsize=11)
for s in ['top','right']: ax.spines[s].set_visible(False)
for s in ['bottom','left']: ax.spines[s].set_color('#333')
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves_compare.png", dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print("✅ Courbes d'entraînement sauvegardées !")


## 💾 Sauvegarde finale

In [ ]:
# Sauvegarde métriques JSON
metrics_json = {
    "PPO_V2": {k: float(v) if isinstance(v, (np.floating, float)) else str(v)
               for k, v in results_ppo.items() if k not in ["env","value_history","trade_history","daily_returns"]},
    "SAC":    {k: float(v) if isinstance(v, (np.floating, float)) else str(v)
               for k, v in results_sac.items() if k not in ["env","value_history","trade_history","daily_returns"]},
    "TD3":    {k: float(v) if isinstance(v, (np.floating, float)) else str(v)
               for k, v in results_td3.items() if k not in ["env","value_history","trade_history","daily_returns"]},
    "BuyHold":{"return": float(bnh_ret), "sharpe": float(bnh_sharpe), "mdd": float(bnh_mdd)},
}
with open(f"{SAVE_DIR}/metrics_v2.json", "w") as f:
    json.dump(metrics_json, f, indent=2)

print("✅ Tous les fichiers sauvegardés dans Drive :")
print(f"   📁 {SAVE_DIR}/")
print(f"   ├── raw_data_v2.csv")
print(f"   ├── features_v2.csv            ({X.shape[1]} features dont sentiment)")
print(f"   ├── vae_lstm_model.pth          VAE-LSTM entraîné")
print(f"   ├── ppo_v2.pth                  Agent PPO V2")
print(f"   ├── sac_v2.pth                  Agent SAC")
print(f"   ├── td3_v2.pth                  Agent TD3")
print(f"   ├── comparaison_finale.csv      Tableau des métriques")
print(f"   ├── dashboard_comparatif.png    Dashboard final")
print(f"   └── training_curves_compare.png Courbes d'entraînement")

print(f"\n{'='*60}")
print(f"  🎉 PROJET V2 COMPLET ET TERMINÉ !")
print(f"{'='*60}")

# Meilleur agent
best_agent = max([results_ppo, results_sac, results_td3], key=lambda x: x["sharpe"])
print(f"\n  🏆 Meilleur agent (Sharpe) : {best_agent['label']}")
print(f"     Rendement  : {best_agent['return']:+.2f}%")
print(f"     Sharpe     : {best_agent['sharpe']:.3f}")
print(f"     Drawdown   : {best_agent['mdd']:.2f}%")
print(f"     Win Rate   : {best_agent['wr']:.1f}%")
